# Tarea 02 - Algoritmo Genético Simple (AGS)



---

### Especificaciones del AGS

| Característica | Valor |
|---|---|
| Representación | Binaria |
| Población | Constante n |
| Inicialización | Aleatoria |
| Renormalización | Ninguna |
| Selección | Rueda de ruleta |
| Operadores | Cruce un punto + Mutación |
| Sustitución | Total |
| Generaciones | 100 |

---
## 0. Importaciones y configuración general

In [2]:
!pip install pygad

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 kB 16.8 MB/s eta 0:00:00


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pygad
import math
import warnings
warnings.filterwarnings("ignore")

plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120



---
## 1. Implementación del AGS para F(x) = x², x entero [0, 63]

Parámetros: n ≈ 20, Pm ≈ 0.002, Pc ≈ 0.8, Generaciones = 100

Representación binaria con **6 bits** (2⁶ = 64 > 63).

In [ ]:
NUM_GENES = 6

def binary_to_int(chromosome):
    x = 0
    for i, bit in enumerate(chromosome):
        x += bit * (2 ** (NUM_GENES - 1 - i))
    return int(x)


def fitness_func_integer(ga_instance, solution, solution_idx):
    x = binary_to_int(solution)
    return x ** 2


def custom_mutation(offspring, ga_instance):
    pm = ga_instance.mutation_probability
    if pm is None:
        pm = 0.002
    for chromosome_idx in range(offspring.shape[0]):
        for gene_idx in range(offspring.shape[1]):
            if np.random.random() < pm:
                offspring[chromosome_idx, gene_idx] = 1 - offspring[chromosome_idx, gene_idx]
    return offspring


def on_generation(ga_instance):
    best_fitness = ga_instance.best_solution()[1]
    ga_instance.best_fitness_history.append(best_fitness)

    fitness = ga_instance.last_generation_fitness
    avg_fitness = np.mean(fitness)
    ga_instance.avg_fitness_history.append(avg_fitness)


def run_aga(pop_size=20, num_generations=100, pc=0.8, pm=0.002, num_genes=6,
            fitness_func=None, on_gen_callback=None, seed=None):

    if fitness_func is None:
        fitness_func = fitness_func_integer
    if on_gen_callback is None:
        on_gen_callback = on_generation

    if seed is not None:
        np.random.seed(seed)

    num_parents_mating = pop_size // 2

    ga_instance = pygad.GA(
        num_generations=num_generations,
        num_parents_mating=num_parents_mating,
        sol_per_pop=pop_size,
        num_genes=num_genes,
        fitness_func=fitness_func,
        gene_type=int,
        gene_space=[0, 1],
        init_range_low=0,
        init_range_high=2,
        parent_selection_type="rws",
        crossover_type="single_point",
        crossover_probability=pc,
        mutation_type=custom_mutation,
        mutation_probability=pm,
        keep_parents=0,
        keep_elitism=0,
        on_generation=on_gen_callback,
        allow_duplicate_genes=True,
        suppress_warnings=True,
        random_seed=seed,
    )

    ga_instance.best_fitness_history = []
    ga_instance.avg_fitness_history = []

    ga_instance.run()

    return (ga_instance,
            ga_instance.best_fitness_history,
            ga_instance.avg_fitness_history)


print("Funciones del AGS definidas correctamente.")
print(f"  Número de bits para x entero [0,63]: {NUM_GENES}")
print(f"  Rango representable: [0, {2**NUM_GENES - 1}]")

---
## 1.1 Medir la ejecución del AGS

Graficar:
- **a)** La adaptación del mejor individuo de la población
- **b)** El promedio de las adaptaciones de los individuos en la población

Parámetros: n=20, Pc=0.8, Pm=0.002, 100 generaciones

In [ ]:
ga_base, best_hist_base, avg_hist_base = run_aga(
    pop_size=20, num_generations=100, pc=0.8, pm=0.002, seed=42
)

best_solution = ga_base.best_solution()
best_chromosome = best_solution[0]
best_x = binary_to_int(best_chromosome)
best_fitness = best_solution[1]

print("=" * 60)
print("PARTE 1.1: Resultados del AGS")
print("=" * 60)
print(f"Parámetros: n=20, Pc=0.8, Pm=0.002, Generaciones=100")
print(f"\nMejor solución encontrada:")
print(f"  Cromosoma: {best_chromosome}")
print(f"  x = {best_x}")
print(f"  F(x) = x² = {best_fitness}")
print(f"  Máximo teórico: x=63, F(63)={63**2}")

In [ ]:
%matplotlib inline

fig, ax = plt.subplots(figsize=(10, 6))
generations = list(range(1, len(best_hist_base) + 1))
ax.plot(generations, best_hist_base, 'b-', linewidth=2, label='a) Mejor individuo')
ax.plot(generations, avg_hist_base, 'r-', linewidth=2, label='b) Promedio de la población')
ax.set_xlabel('Generación', fontsize=12)
ax.set_ylabel('Aptitud (F(x) = x²)', fontsize=12)
ax.set_title('Parte 1.1: Evolución del AGS\n(n=20, Pc=0.8, Pm=0.002)', fontsize=14)
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 1.2 Efecto de variar Pc (Pm=0.002 fijo)

Para Pm=0.002, ¿qué cambios experimentan los observables **a** y **b** si fijamos:
- Pc=0.6
- Pc=0.4

In [ ]:
pc_values = [0.8, 0.6, 0.4]
results_1_2 = {}

for pc in pc_values:
    ga, best_h, avg_h = run_aga(pop_size=20, num_generations=100, pc=pc, pm=0.002, seed=42)
    results_1_2[pc] = {'best': best_h, 'avg': avg_h, 'ga': ga}
    best_sol = ga.best_solution()
    best_x_val = binary_to_int(best_sol[0])
    print(f"  Pc={pc}: Mejor x={best_x_val}, F(x)={best_sol[1]:.0f}, "
          f"Mejor aptitud final={best_h[-1]:.0f}, Promedio final={avg_h[-1]:.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for pc in pc_values:
    axes[0].plot(range(1, len(results_1_2[pc]['best']) + 1),
                 results_1_2[pc]['best'], linewidth=2, label=f'Pc={pc}')
axes[0].set_xlabel('Generación', fontsize=12)
axes[0].set_ylabel('Aptitud del mejor individuo', fontsize=12)
axes[0].set_title('a) Mejor individuo (Pm=0.002)', fontsize=13)
axes[0].legend(loc='best', fontsize=10)
axes[0].grid(True, alpha=0.3)

# Promedio (observable b)
for pc in pc_values:
    axes[1].plot(range(1, len(results_1_2[pc]['avg']) + 1),
                 results_1_2[pc]['avg'], linewidth=2, label=f'Pc={pc}')
axes[1].set_xlabel('Generación', fontsize=12)
axes[1].set_ylabel('Aptitud promedio', fontsize=12)
axes[1].set_title('b) Promedio de la población (Pm=0.002)', fontsize=13)
axes[1].legend(loc='best', fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Parte 1.2: Efecto de la Probabilidad de Cruce (Pc)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
print("Análisis de convergencia por Pc:")
print("=" * 60)
for pc in pc_values:
    r = results_1_2[pc]
    final_best = r['best'][-1]
    final_avg = r['avg'][-1]
    conv_gen = None
    for i in range(len(r['best'])):
        if r['best'][i] >= 3969:  # 63² = 3969
            conv_gen = i + 1
            break
    print(f"  Pc={pc}:")
    print(f"    - Mejor aptitud final: {final_best:.0f}")
    print(f"    - Promedio final: {final_avg:.0f}")
    print(f"    - Generación de convergencia al máximo: {conv_gen if conv_gen else 'No convergió'}")
    print()

### Respuesta 1.2

**Al disminuir la probabilidad de cruce Pc:**

- **Pc=0.8** (valor alto): El AG explota eficientemente las buenas soluciones mediante recombinación. La convergencia es rápida tanto para el mejor individuo como para el promedio de la población.

- **Pc=0.6**: Se reduce la tasa de recombinación, lo que implica que una fracción mayor de la población se reproduce sin cruce. Esto ralentiza la convergencia, ya que se generan menos combinaciones nuevas de genes favorables. El mejor individuo puede alcanzar el óptimo, pero el promedio de la población converge más lentamente.

- **Pc=0.4**: La baja probabilidad de cruce significa que la mayoría de los descendientes son copias directas de los padres (sin recombinación). Esto limita significativamente la exploración del espacio de búsqueda. El mejor individuo puede tardar mucho más en aparecer (o no aparecer), y el promedio de la población crece muy lentamente, ya que la diversidad genética solo proviene de la mutación (que con Pm=0.002 es muy baja).

**En resumen:** Reducir Pc frena la convergencia del promedio (observable b) y puede retrasar la aparición del mejor individuo (observable a), porque el cruce es el operador principal que combina material genético favorable de diferentes individuos.

---
## 1.3 Efecto de variar Pm (Pc=0.8 fijo)

Para Pc=0.8, ¿qué cambios experimentan los observables **a** y **b** si fijamos:
- Pm=0.01
- Pm=0.1

In [ ]:
pm_values = [0.002, 0.01, 0.1]
results_1_3 = {}

for pm in pm_values:
    ga, best_h, avg_h = run_aga(pop_size=20, num_generations=100, pc=0.8, pm=pm, seed=42)
    results_1_3[pm] = {'best': best_h, 'avg': avg_h, 'ga': ga}
    best_sol = ga.best_solution()
    best_x_val = binary_to_int(best_sol[0])
    print(f"  Pm={pm}: Mejor x={best_x_val}, F(x)={best_sol[1]:.0f}, "
          f"Mejor aptitud final={best_h[-1]:.0f}, Promedio final={avg_h[-1]:.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for pm in pm_values:
    axes[0].plot(range(1, len(results_1_3[pm]['best']) + 1),
                 results_1_3[pm]['best'], linewidth=2, label=f'Pm={pm}')
axes[0].set_xlabel('Generación', fontsize=12)
axes[0].set_ylabel('Aptitud del mejor individuo', fontsize=12)
axes[0].set_title('a) Mejor individuo (Pc=0.8)', fontsize=13)
axes[0].legend(loc='best', fontsize=10)
axes[0].grid(True, alpha=0.3)

for pm in pm_values:
    axes[1].plot(range(1, len(results_1_3[pm]['avg']) + 1),
                 results_1_3[pm]['avg'], linewidth=2, label=f'Pm={pm}')
axes[1].set_xlabel('Generación', fontsize=12)
axes[1].set_ylabel('Aptitud promedio', fontsize=12)
axes[1].set_title('b) Promedio de la población (Pc=0.8)', fontsize=13)
axes[1].legend(loc='best', fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Parte 1.3: Efecto de la Probabilidad de Mutación (Pm)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
print("Análisis de resultados por Pm:")
print("=" * 60)
for pm in pm_values:
    r = results_1_3[pm]
    final_best = r['best'][-1]
    final_avg = r['avg'][-1]
    print(f"  Pm={pm}:")
    print(f"    - Mejor aptitud final: {final_best:.0f}")
    print(f"    - Promedio final: {final_avg:.0f}")
    print()

### Respuesta 1.3

**Al aumentar la probabilidad de mutación Pm:**

- **Pm=0.002** (valor bajo): La mutación introduce muy poca diversidad. El AG converge rápidamente hacia el óptimo gracias al cruce, y una vez que la población se homogeneiza, el mejor individuo se mantiene estable. Sin embargo, la baja diversidad puede causar que el AG se quede atrapado en un subóptimo si la población converge prematuramente antes de descubrir el cromosoma óptimo.

- **Pm=0.01**: La mayor tasa de mutación introduce más diversidad genética. Esto puede ser beneficioso en las generaciones iniciales (mayor exploración), ayudando a escapar de subóptimos. Sin embargo, en las generaciones finales puede causar oscilaciones: el mejor individuo puede alcanzar el óptimo pero luego ser "destruido" por mutaciones, y el promedio de la población muestra más fluctuaciones.

- **Pm=0.1** (valor alto): La alta tasa de mutación convierte el AG esencialmente en una búsqueda aleatoria. La diversidad es tan alta que la presión selectiva no puede concentrar a la población alrededor del óptimo. El mejor individuo (observable a) oscila erráticamente, mientras que el promedio (observable b) se mantiene bajo y con grandes fluctuaciones, sin convergencia.

**En resumen:** Aumentar Pm más allá del valor óptimo destruye la convergencia. El observable a (mejor individuo) se vuelve inestable, y el observable b (promedio) no logra converger, mostrando que una mutación excesiva impide la explotación de buenas soluciones.

---
## 1.4 F(x) = x², x real [0, 63] con precisión de 2 cifras decimales

Tamaño de la población = 50.

Preguntas:
1. ¿Qué cambio debo realizar en el código del AGS?
2. ¿En cuántas partes debo dividir el intervalo [0, 63]?
3. ¿Cuál es el largo del cromosoma o número de bits?

In [ ]:
precision = 0.01
a, b = 0.0, 63.0
num_intervals = (b - a) / precision
num_values = num_intervals + 1
num_bits_real = math.ceil(math.log2(num_values))

print("Cálculos para representación con precisión de 2 decimales")
print("=" * 60)
print(f"  Intervalo: [{a}, {b}]")
print(f"  Precisión requerida: {precision} (2 cifras decimales)")
print(f"  Número de subintervalos: (b-a)/precision = ({b}-{a})/{precision} = {num_intervals:.0f}")
print(f"  Número de valores posibles: {num_values:.0f}")
print(f"  Bits necesarios: ceil(log2({num_values:.0f})) = ceil({math.log2(num_values):.4f}) = {num_bits_real}")
print(f"  2^{num_bits_real} = {2**num_bits_real} valores representables")
resolution = (b - a) / (2**num_bits_real - 1)
print(f"  Resolución real con {num_bits_real} bits: ({b}-{a})/(2^{num_bits_real}-1) = {resolution:.6f}")
print(f"  Esta resolución ({resolution:.6f}) es < {precision}? → {'SÍ' if resolution < precision else 'NO'}, se cumple la precisión requerida")

In [ ]:
NUM_GENES_REAL = num_bits_real

def binary_to_real(chromosome):
    int_val = 0
    for i, bit in enumerate(chromosome):
        int_val += bit * (2 ** (NUM_GENES_REAL - 1 - i))
    x_real = a + int_val * (b - a) / (2 ** NUM_GENES_REAL - 1)
    return round(x_real, 2)


def fitness_func_real(ga_instance, solution, solution_idx):
    x = binary_to_real(solution)
    return x ** 2


def custom_mutation_real(offspring, ga_instance):
    pm = ga_instance.mutation_probability
    if pm is None:
        pm = 0.002
    for chromosome_idx in range(offspring.shape[0]):
        for gene_idx in range(offspring.shape[1]):
            if np.random.random() < pm:
                offspring[chromosome_idx, gene_idx] = 1 - offspring[chromosome_idx, gene_idx]
    return offspring


def on_generation_real(ga_instance):
    best_fitness = ga_instance.best_solution()[1]
    ga_instance.best_fitness_history.append(best_fitness)
    fitness = ga_instance.last_generation_fitness
    avg_fitness = np.mean(fitness)
    ga_instance.avg_fitness_history.append(avg_fitness)


print("Funciones para representación real definidas correctamente.")
print(f"  Número de bits para x real [0,63] con 2 decimales: {NUM_GENES_REAL}")

In [ ]:
ga_real = pygad.GA(
    num_generations=100,
    num_parents_mating=25,
    sol_per_pop=50,
    num_genes=NUM_GENES_REAL,
    fitness_func=fitness_func_real,
    gene_type=int,
    gene_space=[0, 1],
    init_range_low=0,
    init_range_high=2,
    parent_selection_type="rws",
    crossover_type="single_point",
    crossover_probability=0.8,
    mutation_type=custom_mutation_real,
    mutation_probability=0.002,
    keep_parents=0,
    keep_elitism=0,
    on_generation=on_generation_real,
    allow_duplicate_genes=True,
    suppress_warnings=True,
    random_seed=42,
)

ga_real.best_fitness_history = []
ga_real.avg_fitness_history = []
ga_real.run()

best_sol_real = ga_real.best_solution()
best_chrom_real = best_sol_real[0]
best_x_real = binary_to_real(best_chrom_real)
best_fitness_real = best_sol_real[1]

print("Resultados AG para x real [0, 63] con 2 decimales:")
print("=" * 60)
print(f"  Cromosoma: {best_chrom_real}")
int_decoded = sum(int(b)*(2**(NUM_GENES_REAL-1-i)) for i,b in enumerate(best_chrom_real))
print(f"  Valor entero decodificado: {int_decoded}")
print(f"  x = {best_x_real}")
print(f"  F(x) = x² = {best_fitness_real:.2f}")
print(f"  Máximo teórico: x=63.00, F(63)={63**2}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
generations_real = list(range(1, len(ga_real.best_fitness_history) + 1))
ax.plot(generations_real, ga_real.best_fitness_history, 'b-', linewidth=2, label='Mejor individuo')
ax.plot(generations_real, ga_real.avg_fitness_history, 'r-', linewidth=2, label='Promedio de la población')
ax.set_xlabel('Generación', fontsize=12)
ax.set_ylabel('Aptitud (F(x) = x²)', fontsize=12)
ax.set_title('Parte 1.4: Evolución del AGS para x real [0, 63]\n(n=50, Pc=0.8, Pm=0.002, 13 bits)', fontsize=14)
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Respuesta 1.4

#### ¿Qué cambio debo realizar en el código del AGS implementado previamente?

Se debe cambiar la **función de decodificación** del cromosoma binario. En lugar de interpretar directamente los bits como un número entero, ahora se debe:

1. Decodificar el cromosoma binario a un valor entero (como antes).
2. Mapear ese valor entero al intervalo real [0, 63] usando la fórmula:
   $$x_{real} = a + \text{valor\_entero} \times \frac{b - a}{2^L - 1}$$
   donde $a=0$, $b=63$ y $L$ es el número de bits del cromosoma.
3. Redondear el resultado a 2 cifras decimales.

Además, se debe aumentar el número de bits del cromosoma ($L$) para representar la mayor cantidad de valores discretos necesarios, y ajustar el tamaño de la población a 50 como indica el enunciado.

#### ¿En cuántas partes debo dividir el intervalo [0, 63]?

Con precisión de 0.01 (2 cifras decimales):
- Número de subintervalos = $(b - a) / \text{precisión} = 63 / 0.01 = 6300$
- Número de valores posibles = 6301

#### ¿Cuál es el largo del cromosoma o número de bits?

$$L = \lceil \log_2(6301) \rceil = \lceil 12.6214 \rceil = 13 \text{ bits}$$

Con 13 bits se pueden representar $2^{13} = 8192$ valores, lo cual es suficiente para los 6301 valores requeridos.

La resolución real obtenida es:
$$\Delta x = \frac{63}{2^{13} - 1} \approx 0.0077 < 0.01$$

Como $\Delta x < 0.01$, se cumple la precisión de 2 decimales.